# LangChain 05 · 多 Agent（官方五种模式）

这一课把「一个 Agent」扩展到「多个 Agent 怎么协作」。LangChain 官方把多 Agent
归纳为**五种模式**，本 notebook 正好一一对应，串成一条从「最简单」到「最灵活」的线：

| 模式 | 一句话 | 控制权怎么流转 | 源文件 |
|---|---|---|---|
| ① 子代理 Subagents | 一个 Agent 当另一个 Agent 的**工具** | 主 Agent 调子 Agent | `12_多Agent_MCP子Agent(.py/_jxsd.py)` |
| ② 交接 Handoffs | 把对话**控制权**交给别的 Agent | 分诊台 / 转接工具 | `13_多Agent_交接(.py/_jxsd.py)` |
| ③ 路由 Router | 按任务拆给多个专家**并行**处理 | 分类器 + 条件边扇出 | `14_多Agent_路由与合并(.py/_jxsd.py)` |
| ④ 技能 Skills | 提示词**按需加载**（渐进披露） | 模型自己决定加载哪个技能 | `17_Skills渐进披露_官方补充.py` |
| ⑤ 自定义工作流 Custom-workflow | Agent 与确定性步骤**混编** | 把 Agent 当图里的节点 | `18_自定义工作流_官方补充.py` |

> **本 notebook 由 `Agent/02_langchain/` 下 8 个脚本合并而成**：
> 12/13/14 各含「课案原版（.py）+ 完整版（_jxsd.py）」两份，
> 17/18 是官方文档补充篇（对应官方 multi-agent 的 skills / custom-workflow 两节）。

**官方文档**
- 多 Agent 总览：<https://docs.langchain.com/oss/python/langchain/multi-agent>
- 技能渐进披露：<https://docs.langchain.com/oss/python/langchain/multi-agent/skills>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型 |
| 依赖 | `langchain` / `langgraph` / `langchain-mcp-adapters`（venv 已装） |
| 密钥 | `settings.api_key`（已配置）；`baidu_qfan_api_key` / `gitee_api_key` 缺了就降级到本地假工具 |
| 前置服务 | 无（真 MCP 走不通时自动降级到本地假工具，不需要你起服务） |
| 预计耗时 | 约 1~2 分钟（几十次模型调用） |

> 本课是 `02_langchain` 里的「需模型」档（🟡），不是离线档：五种模式的每一条
> 链路都要真实调用大模型。里面两处会走到**降级分支**（第 2、3 节的思考模式限制），
> 源文件都已内置好降级，本 notebook 原样保留并写明原因。

## 本节地图

五种模式按「控制权怎么流转」从简单到灵活排列：

```mermaid
graph TD
    A["多 Agent 五种模式"] --> B["① 子代理<br/>一个 Agent = 另一个的工具"]
    A --> C["② 交接<br/>把对话控制权交出去"]
    A --> D["③ 路由<br/>并行分发给多个专家"]
    A --> E["④ 技能<br/>提示词按需加载"]
    A --> F["⑤ 自定义工作流<br/>Agent 混编确定性步骤"]
    B -.-> C
    C -.-> D
    D -.-> E
    E -.-> F
```

等价表格（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 顺序 | 模式 | 核心 API / 手法 | 关键区别 |
|---|---|---|---|
| ① | 子代理 | `@tool` 里 `subagent.ainvoke(...)` | 子 Agent 的答复就是一坨文本 → 天然可当工具返回值 |
| ② | 交接 | `Command(goto=..., graph=Command.PARENT)` | 转接工具返回「控制指令」，把说话权交给别人 |
| ③ | 路由 | `add_conditional_edges` + `Send` | `Send` 列表 = 一次并行派发多个节点 |
| ④ | 技能 | 手写 `load_skill` 工具 | 技能正文只在模型决定要用它之后才进上下文 |
| ⑤ | 自定义工作流 | `StateGraph` 里 `agent.invoke(...)` | Agent 不再是终点，只是流程里的一环 |

与上一课 `04_中间件_钩子与人工审核` 的关系：那课把单个 Agent 的「内部」讲透了
（钩子 / 中间件 / 人工审核）；这一课跳到「外部」——多个 Agent 之间怎么组合。
其中 ② 交接、③ 路由、⑤ 自定义工作流都直接落在 `01_langgraph` 学过的图 API 上。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

它还顺带给出 `NB_DIR` / `ROOT` / `WORKDIR` 三个变量：本课不往磁盘写文件，
所以 `WORKDIR` 用不上，但保留它保持全仓统一。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 前置条件自检

五种模式都要调用模型；第 1、3 节还会涉及 MCP 密钥。先把现状打印出来，
缺了哪个后面就会自动走降级分支（不会崩），而不是跑到一半才发现。

In [ ]:
# ===== 前置条件自检（缺了就打印中文提示，不抛异常）=====
from config import settings

print("模型：", settings.model_name, "@", settings.base_url)
print("模型密钥：", "已配置" if settings.api_key else "未配置（后续模型调用都会失败）")
print("百度千帆密钥（联网搜索 MCP）：",
      "已配置" if settings.baidu_qfan_api_key else "未配置（第 1/3 节会降级到本地假工具）")
print("Gitee 密钥（代码仓库 MCP）：",
      "已配置" if settings.gitee_api_key else "未配置（第 3 节会降级到本地假工具）")

## 1. 子代理（Subagents）：一个 Agent 当另一个 Agent 的工具

这是五种模式里**最朴素**的一种，也是最容易被小看的一种。核心洞察只有一句话：

> 工具的返回值对模型来说就是一坨文本，而「子 Agent 的最终答复」**恰好也是一坨文本**——
> 所以「一个 Agent」天然可以当作「另一个 Agent 的工具」。这就是 subagent / supervisor
> 模式的最小实现，不需要框架引入任何新概念。

课案里的子 Agent 通过 **MCP（Model Context Protocol）** 接入工具服务器：
`MultiServerMCPClient` 把 MCP 服务端暴露的工具**转换成 LangChain 工具对象**，
之后 `create_agent` 完全不管这些工具是本地函数还是远程服务。transport 有两种：

- `streamable-http`：连一个已经在跑的远程 MCP 服务（完整版用这种）；
- `stdio`：由适配器**拉起本地子进程**当 MCP 服务（原版用这种）。

### 1.1 课案原版：MCP 子智能体（stdio 最短写法）

原版最短，只用 `stdio` 方式接一个天气 MCP 服务，再把「天气专家」包成主 Agent 的工具。
注意下面 `ask_weather_agent` 那个 `@tool`：它就是「一个 Agent 当另一个 Agent 的工具」
的全部秘密——工具函数内部 `await weather_agent.ainvoke(...)`，把子 Agent 的答复
`return` 出去当工具结果。

> ⚠️ **本节只展示定义、不实际运行 `main()`**：`stdio` 会 `uv run` 拉起 `05_mcp` 章的
> 真实 MCP 服务端子进程，那是「服务档」的职责，本课（需模型档）不启动它。
> 主/子 Agent 结构的**可运行演示**见下一节 1.2（完整版用本地假工具替身，接口一模一样）。

In [ ]:
# ---------- 1.1 课案原版：MCP 子智能体（stdio 最短写法） ----------
# 一个主 Agent 下挂多个「领域专家」子 Agent，每个子 Agent 通过 MCP 接入不同工具服务器。
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# MCP 服务端脚本路径：原版用 Path(__file__).resolve().parents[1] 定位到 05_mcp 章的服务端，
# notebook 里没有 __file__，这里基于仓库根定位（本节只展示定义，不实际启动 stdio MCP 服务）。
from pathlib import Path  # noqa: E402

MCP_SERVER = str(ROOT / "Agent" / "05_mcp" / "01_服务端.py")

In [ ]:
async def main():
    # ---------- 1. 连接 MCP 服务，加载工具 ----------
    # 0.3 版本不再支持 async with 上下文管理器，直接实例化
    client = MultiServerMCPClient(
        {
            # 方式 A：stdio——由适配器自动拉起本地 MCP 服务子进程
            "weather": {
                "command": "uv",
                "args": ["run", MCP_SERVER],
                "transport": "stdio",
            },
            # 方式 B：http——连接已经在运行的服务
            # "weather": {
            #     "url": "http://127.0.0.1:8000/mcp",
            #     "transport": "streamable_http",
            # },
        }
    )
    weather_tools = await client.get_tools()  # 把 MCP 工具转成 LangChain 工具

    # ---------- 2. 每个领域一个子 Agent ----------
    weather_agent = create_agent(
        model=llm,
        tools=weather_tools,
        system_prompt="你是天气专家，只负责回答天气相关的问题。",
    )

    # ---------- 3. 主 Agent：决定找哪个专家 ----------
    # 把子 Agent 包装成一个普通工具（接收问题字符串，返回专家的回答）
    from langchain_core.tools import tool  # noqa: E402

    @tool
    async def ask_weather_agent(question: str) -> str:
        """咨询天气专家。question：要问的问题"""
        # MCP 工具是异步的，内部子 Agent 必须用 ainvoke
        result = await weather_agent.ainvoke({"messages": [("user", question)]})
        return result["messages"][-1].content

    main_agent = create_agent(
        model=llm,
        tools=[ask_weather_agent],
        system_prompt="你是总管，天气问题交给 ask_weather_agent 工具处理。",
    )

    result = await main_agent.ainvoke(
        {"messages": [("user", "上海适合穿短袖吗？")]}
    )
    print("AI：", result["messages"][-1].content)

### 1.2 完整版：百度千帆 MCP + 本地假工具降级

完整版把「原版用 stdio 接本地服务」换成「用 streamable-http 接**远程**百度千帆联网搜索 MCP」，
并补上一整套工程化：

1. `preflight()` 先查依赖和密钥；
2. 密钥连不上时打印**中文提示**，改走一条**本地假搜索工具**的降级路径——
   主 Agent + 子 Agent 的结构、`research` 工具的写法**完全一样**，只是把
   「远程 MCP 工具」换成「本地函数工具」，不抛异常；
3. 密钥一旦填上就自动切回真 MCP（`FORCE_LOCAL_DEMO` 保持 False 即可）。

先看「真 MCP 路径」的代码（下面这一段本机大概率走不到，但课案原文要完整保留）。

In [ ]:
# ---------- 1.2a 依赖与密钥：try/except 兜住缺包 ----------
import asyncio

# notebook 特有：内核里已有一个运行中的事件循环，裸 asyncio.run 会抛 RuntimeError；
# nest_asyncio.apply() 之后 asyncio.run 才能用（本课两处 async main 都需要）。
import nest_asyncio

nest_asyncio.apply()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from config import settings

# MCP 适配器与 mcp 包：本项目已装（langchain-mcp-adapters 0.3.2 / mcp 1.30.0）。
# 按规范用 try/except 兜住缺包的情况，保证模块层面永远 import 成功。
try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    MCP_IMPORT_ERROR = None
except ImportError as exc:                      # pragma: no cover - 取决于本机环境
    MultiServerMCPClient = None
    MCP_IMPORT_ERROR = exc

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# 百度千帆「联网搜索 MCP」的服务地址（课案原文）
BAIDU_MCP_URL = "https://qianfan.baidubce.com/v2/tools/web-search/mcp"

# 强制降级开关：False 表示「密钥非空就优先连真 MCP」
FORCE_LOCAL_DEMO = False

In [ ]:
# ---------- 1.2b 前置检查：依赖 + 密钥 ----------
def preflight() -> tuple[bool, str]:
    """返回 (能否走真 MCP, 中文说明)。"""
    if MCP_IMPORT_ERROR is not None:
        return False, (
            "未安装 MCP 适配器：请先执行\n"
            '    uv add "mcp>=1.9,<2.0"\n'
            "    uv add langchain_mcp_adapters"
        )
    if not settings.baidu_qfan_api_key:
        return False, (
            "settings.baidu_qfan_api_key 为空（百度千帆联网搜索 MCP 需要它）。\n"
            "    申请地址：https://console.bce.baidu.com/qianfan/tools/toolsCenter/"
            "57d4e765-8af5-4ec0-8f9b-47075ec349e0/detail\n"
            "    配置方式：在项目根目录 .env 里加一行 BAIDU_QFAN_API_KEY=<你的密钥>\n"
            "    本节将改用「本地假搜索工具」演示同样的主/子 Agent 结构。"
        )
    return True, "已检测到百度千帆密钥，将直连远程 MCP 服务。"

In [ ]:
# ---------- 1.2c 真 MCP 路径（课案原文，密钥来自 settings，不硬编码） ----------
async def create_research_agent_with_mcp():
    """创建带有搜索 MCP 工具的调研 agent"""

    # 创建 MCP 客户端，加载搜索工具
    client = MultiServerMCPClient(
        {
            "web-search-mcp-server": {
                "url": BAIDU_MCP_URL,
                "transport": "streamable-http",
                # 课案原文写的是 "Bearer xxx"（占位符）；这里从配置读，绝不硬编码密钥
                "headers": {
                    "Authorization": f"Bearer {settings.baidu_qfan_api_key}"
                },
            }
        }
    )

    # 获取 MCP 工具：适配器会去服务端 list_tools，并把每个远程工具包成 LangChain 工具
    # —— 所以下一步 create_agent 完全不知道这些工具来自远程服务，接口和本地 @tool 一样。
    mcp_tools = await client.get_tools()
    print(f"加载到的 MCP 工具: {[t.name for t in mcp_tools]}")

    # 直接调用工具测试（课案的原文就是先单测一个工具，确认连通了再交给 Agent）
    test_result = await mcp_tools[0].ainvoke({"query": "百度"})
    print("MCP 工具直调结果：", str(test_result)[:200])

    # 创建 subagent，将 MCP 工具添加进去
    subagent = create_agent(model=llm, tools=mcp_tools)
    return subagent

In [ ]:
# ---------- 1.2d 降级路径：本地假搜索工具（结构完全一样，只是工具来源换成函数） ----------
@tool
async def local_web_search(query: str) -> str:
    """联网搜索。query：要搜索的问题。

    说明：这是**降级演示用的本地假工具**，不联网，返回写死的示例资料。
    真跑通远程搜索需要百度千帆 MCP 密钥，见文件头的前置条件。
    """
    # 故意做成 async：与 MCP 工具的调用方式保持一致（MCP 工具都是异步的）
    fake_db = {
        "mcp": (
            "【资料】MCP（Model Context Protocol，模型上下文协议）是 Anthropic 于 2024 年底"
            "提出的开放协议，用统一的方式把「外部工具/数据源」接给大模型应用。"
            "它把集成方拆成 MCP 客户端（Agent 侧）与 MCP 服务端（工具侧），"
            "服务端暴露 tools / resources / prompts 三类能力，客户端按协议发现并调用。"
            "好处是工具只需实现一次，任何支持 MCP 的客户端都能复用。"
        ),
        "langchain": (
            "【资料】LangChain 1.x 用 create_agent 封装 ReAct 循环，"
            "并用 middleware 机制提供钩子与内置中间件（摘要、重试、限流、人工审核等）。"
        ),
    }
    key = "mcp" if "mcp" in query.lower() else "langchain"
    print(f"  [local_web_search] 收到查询：{query}")
    return fake_db[key]


async def create_research_agent_local():
    """降级版调研 agent：把远程 MCP 工具换成同名的本地工具。"""
    subagent = create_agent(
        model=llm,
        tools=[local_web_search],
        system_prompt="你是检索助手，先调用 local_web_search 获取资料，再用简洁的中文总结。",
    )
    return subagent

In [ ]:
# ---------- 1.2e 主流程（课案原文的主/子 Agent 组装方式，两种来源共用） ----------
async def main() -> None:
    # ---------- 3.1 前置检查 ----------
    can_use_mcp, message = preflight()
    print("=" * 70)
    print("前置检查：", "通过" if can_use_mcp else "未通过")
    print(message)
    print("=" * 70)

    # ---------- 3.2 挑一条路径建子 Agent ----------
    if can_use_mcp and not FORCE_LOCAL_DEMO:
        try:
            subagent = await create_research_agent_with_mcp()
            print("已通过百度千帆 MCP 加载搜索工具。\n")
        except Exception as exc:
            # 网络不通 / 密钥失效 / 服务端报错，都不该让教学脚本崩掉
            print(f"[降级] 连接百度千帆 MCP 失败：{type(exc).__name__}: {str(exc)[:160]}")
            print("       改用本地假搜索工具演示同样的结构。\n")
            subagent = await create_research_agent_local()
    else:
        subagent = await create_research_agent_local()
        print("使用本地假搜索工具（结构、调用方式与 MCP 版完全一致）。\n")

    # ---------- 3.3 定义调研工具（课案原文写法） ----------
    # 重点：子 Agent 的最终答复就是一坨文本，所以可以直接当成工具返回值。
    # 「一个 Agent 当另一个 Agent 的工具」= subagent 模式的最小实现。
    @tool("research", description="当用户的问题需要查询互联网资料或最新信息时调用。"
          "输入需要调研的问题，工具会使用百度搜索获取资料，"
          "并返回整理后的调研结果。")
    async def call_research_agent(query: str):
        result = await subagent.ainvoke({"messages": [{"role": "user", "content": query}]})
        return result["messages"][-1].content

    # ---------- 3.4 创建主 agent ----------
    # 主 Agent 手里只有一个工具：research。它负责判断「这个问题要不要调研」。
    main_agent = create_agent(model=llm, tools=[call_research_agent])

    # ---------- 3.5 调用主 agent ----------
    question = "大模型的MCP是什么"
    print(f"用户提问：{question}")
    result = await main_agent.ainvoke(
        {"messages": [{"role": "user", "content": question}]}
    )

    # 打印整条链路：主 Agent → research 工具 → 子 Agent → 搜索工具 → …
    print("\n执行轨迹：")
    for index, msg in enumerate(result["messages"], start=1):
        kind = type(msg).__name__
        calls = getattr(msg, "tool_calls", None)
        if calls:
            print(f"  [{index}] {kind:<14} 调用工具 {[c['name'] for c in calls]}")
        else:
            print(f"  [{index}] {kind:<14} {str(msg.content)[:80]}")

    print("\n最终答复：")
    print(result["messages"][-1].content)

In [ ]:
# 源脚本这里是 asyncio.run(main())；内核里已有一个运行中的事件循环，直接跑会 RuntimeError。
# 上面已经 nest_asyncio.apply() 过，所以这里可以原样 asyncio.run(main())。
asyncio.run(main())

### 预期输出

```text
======================================================================
前置检查： 通过
已检测到百度千帆密钥，将直连远程 MCP 服务。
======================================================================
加载到的 MCP 工具: ['webSearch']
MCP 工具直调结果： [{'type': 'text', 'text': 'bad status: 429, body: ...QUOTA_USER_DAILY_FREE...', ...}]
已通过百度千帆 MCP 加载搜索工具。

用户提问：大模型的MCP是什么

执行轨迹：
  [1] HumanMessage   大模型的MCP是什么
  [2] AIMessage      调用工具 ['research', 'research']
  [3] ToolMessage    ...（子 Agent 的答复）
  ...

最终答复：
...（模型整理出的 MCP 介绍）
```

> ⚠️ 模型措辞与「走真 MCP 还是降级」都**每次可能不同**：本机密钥现在**有效**，
> 所以走的是**真 MCP**（`get_tools()` 拿到 `webSearch`，直调时百度返回了
> `429 QUOTA_USER_DAILY_FREE`——每日免费配额用完，作为**文本**返回而不是抛异常）。
> 密钥失效 / 连不上时，会打印 `[降级] 连接百度千帆 MCP 失败` 并改用本地假工具；
> 两种情况都在设计内、不抛异常。上面是实测值。

## 2. 交接（Handoffs）：把对话控制权交出去

子代理模式里，主 Agent **始终在场**、由它把结果拼起来。交接模式要解决的是另一个问题：
**当前 Agent 怎么把「接下来谁说话」的决定权交出去？**——像客服把电话转给销售，
转出去之后，客服就不再参与后面的对话。

课案给了两种实现，正好是「同一件事的两代写法」：
- 原版：**分诊台节点 + 条件边**，用图结构实现控制权移交（LangGraph 手写）；
- 完整版：**转接工具 + `Command(goto=..., graph=Command.PARENT)`**，
  让模型自己决定调转接工具，把控制权从子图冒泡回父图。

### 2.1 课案原版：分诊台 + 条件边

原版把每个专家做成图上的节点，用一个「分诊台（supervisor）」节点判断问题属于哪个领域，
再用条件边把控制权交给对应专家。它等效于 handoff 工具，但把「谁决定、怎么跳」
都显式摆在图里，最适合理解交接的**决策点**在哪。

原版里的专家用的是 `create_react_agent`（LangGraph prebuilt 的经典写法）。

In [ ]:
# ---------- 2.1a 课案原版：交接（分诊台 + 条件边） ----------
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import create_react_agent
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# ---------- 各自的工具 ----------
@tool
def query_bill() -> str:
    """查询用户账单"""
    return "本月账单：话费 99 元"


@tool
def repair_network() -> str:
    """远程检修网络"""
    return "网络已重置，恢复正常"

In [ ]:
# ---------- 两个专家（各自带工具的智能体） ----------
billing_agent = create_react_agent(
    llm,
    [query_bill],
    prompt="你是账务客服，只处理账单问题，回答要简短。",
)

network_agent = create_react_agent(
    llm,
    [repair_network],
    prompt="你是网络客服，只处理网络问题，回答要简短。",
)

# ---------- 分诊台：决定把对话移交给谁 ----------
def supervisor(state: MessagesState):
    """前台：只判断交给谁（这一步就是「交接」的决策点）"""
    response = llm.invoke(
        [
            ("system", "你是分诊台。判断用户问题属于哪类，只回复一个词：billing（账务）或 network（网络）"),
            *state["messages"],
        ]
    )
    target = "billing_agent" if "billing" in response.content.lower() else "network_agent"
    print(f"[分诊台] 交接给 -> {target}")
    return {"messages": [response]}

In [ ]:
def _run_specialist(agent, state: MessagesState):
    """把整段对话交给专家处理；端点不兼容时退回「只交用户消息」再试一次。

    ⚠️ 实测踩坑（2026-09，端点=DeepSeek 思考模型）：思考模式要求 assistant 消息把
    `reasoning_content` **一起回传**，而分诊台那条 AIMessage 经 LangGraph 状态往返后
    该字段丢了，专家代理再把它发给模型就会 400：
        The `reasoning_content` in the thinking mode must be passed back to the API.
    退回策略：摘掉分诊台那条 AIMessage，只把用户消息交给专家 ——
    这也更贴近「交接到专家」的真实语义（分诊台的内部判断不必进专家的上下文）。
    """
    try:
        return agent.invoke({"messages": state["messages"]})
    except Exception as exc:   # noqa: BLE001 —— 端点不支持回放思考消息时降级，不崩
        trimmed = state["messages"][:-1] or state["messages"]
        print(f"  [降级] 端点拒绝回放分诊台消息（{type(exc).__name__}）：{str(exc)[:90]}")
        print("         改为只把用户消息交给专家（分诊台的判断不进专家上下文）。")
        return agent.invoke({"messages": trimmed})


def handoff_billing(state: MessagesState):
    """交接到账务专家：以独立身份处理整段对话"""
    result = _run_specialist(billing_agent, state)
    return {"messages": [result["messages"][-1]]}


def handoff_network(state: MessagesState):
    """交接到网络专家"""
    result = _run_specialist(network_agent, state)
    return {"messages": [result["messages"][-1]]}

In [ ]:
# ---------- 组装图：分诊台按领域移交控制权 ----------
builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor)
builder.add_node("billing_agent", handoff_billing)
builder.add_node("network_agent", handoff_network)
builder.add_edge(START, "supervisor")
# 条件边：分诊结果决定控制权移交给哪个专家
builder.add_conditional_edges(
    "supervisor",
    lambda s: "billing_agent" if any("billing" in m.content.lower() for m in [s["messages"][-1]]) else "network_agent",
    ["billing_agent", "network_agent"],
)
builder.add_edge("billing_agent", END)
builder.add_edge("network_agent", END)

graph = builder.compile(checkpointer=MemorySaver())

In [ ]:
# 例 1：网络问题 → 移交给网络专家
r1 = graph.invoke(
    {"messages": [("user", "我家断网了，帮我修一下")]},
    config={"configurable": {"thread_id": "ho-1"}},
)
print("最终回复：", r1["messages"][-1].content)

# 例 2：账单问题 → 移交给账务专家
r2 = graph.invoke(
    {"messages": [("user", "帮我查一下这个月话费")]},
    config={"configurable": {"thread_id": "ho-2"}},
)
print("最终回复：", r2["messages"][-1].content)

### 预期输出

```text
[分诊台] 交接给 -> network_agent
  [降级] 端点拒绝回放分诊台消息（OpenAIInvalidRequestError）：Error code: 400 - {'error': {'message': 'The `reasoning_content` in the thinking mode must
         改为只把用户消息交给专家（分诊台的判断不进专家上下文）。
最终回复： 已修复，网络已重置恢复正常。...
[分诊台] 交接给 -> billing_agent
  [降级] 端点拒绝回放分诊台消息（OpenAIInvalidRequestError）：Error code: 400 - ...
         改为只把用户消息交给专家（分诊台的判断不进专家上下文）。
最终回复： 本月话费为 **99 元**。
```

> ⚠️ 模型措辞每次不同；`[分诊台] 交接给 -> ...` 与 `[降级] 端点拒绝回放...` 两行是
> **确定性打印**（400 报错原文被 `str(exc)[:90]` 截断到 90 字符），「最终回复」是模型生成的正文。
> 上面只是我这次跑出来的实测值。

### 2.2 完整版：转接工具 + `Command.PARENT`

完整版是课案的「工具即出口」写法——每个专家手里握一个**转接工具**（`transfer_to_sales`
/ `transfer_to_support`），模型判断「这问题不归我管」时就调它，工具不返回字符串，
而是返回一个 `Command`（LangGraph 的控制指令）：

```python
return Command(goto="sales_agent", update={...}, graph=Command.PARENT)
```

关键在 `graph=Command.PARENT`：`create_agent` 内部封装了一张**子图**，而
`sales_agent` / `support_agent` 是**父图**里的节点。不写 `PARENT`（默认 `LOCAL`），
LangGraph 会到子图里找 `sales_agent` 节点，找不到就报错或静默不跳转——交接就失效了。

两个容易忽略的细节（都写进了下方代码注释）：
- `update={"messages": [last_ai_message, transfer_message]}` 必须**成对**写回：
  只写 ToolMessage 会出现「没有对应 tool_call 的孤儿 ToolMessage」，下次请求直接 400；
- `runtime.tool_call_id` 由框架注入，就是本次调用编号，用它给 `ToolMessage` 配对。

能力边界也由工具决定：销售只拿「转客服」、客服只拿「转销售」，谁也拿不到别人的业务工具，
所以模型只能「自己答」或「转出去」，不会越权调不属于自己的工具。

In [ ]:
# ---------- 2.2a 完整版：交接（转接工具 + Command.PARENT） ----------
from typing import Literal

from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from typing_extensions import NotRequired
from config import settings

# 课案写的是 ChatOpenAI(model=setting.MODEL_NAME, ...)；本项目统一用 init_chat_model + settings
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 1. 定义状态（包含当前活跃智能体的追踪器） ----------
class MultiAgentState(AgentState):
    # AgentState 已经带好了 messages；这里再加一个字段记录「现在是谁在值班」。
    # NotRequired 表示这个键可以不传（第一次进来时还没有活跃 Agent）。
    active_agent: NotRequired[str]

In [ ]:
# ---------- 2. 创建智能体切换工具 ----------
@tool
def transfer_to_sales(
    runtime: ToolRuntime,
) -> Command:
    """将对话转接给销售智能体。"""
    # 找到最后一条AI消息
    # —— 这条就是「模型决定转接」的那条 AIMessage，它带着 tool_calls，
    #    必须和下面的 ToolMessage 一起写回状态，消息序列才完整。
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    # 创建转接通知消息
    transfer_message = ToolMessage(
        content="已从客服智能体转接至销售智能体",
        tool_call_id=runtime.tool_call_id,     # 框架注入的本次调用编号，必须配对
    )
    # 返回跳转命令
    return Command(
        goto="sales_agent",                    # 下一站：父图里的 sales_agent 节点
        update={
            "active_agent": "sales_agent",     # 同步更新「谁在值班」
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,                  # 目标在父图，不在 create_agent 那张子图里
    )


# Command 的 graph 参数有两个主要选项：
# Command.LOCAL（默认）：goto 的目标是当前子图内的节点。
# Command.PARENT：goto 的目标是父图内的节点。
# create_agent 内部实际上封装了一个子图逻辑。


@tool
def transfer_to_support(
    runtime: ToolRuntime,
) -> Command:
    """将对话转接给客服智能体。"""
    # 找到最后一条AI消息
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    # 创建转接通知消息
    transfer_message = ToolMessage(
        content="已从销售智能体转接至客服智能体",
        tool_call_id=runtime.tool_call_id,
    )
    # 返回跳转命令
    return Command(
        goto="support_agent",
        update={
            "active_agent": "support_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )

In [ ]:
# ---------- 3. 创建智能体并绑定切换工具 ----------
# 注意每个 Agent 手里的工具：销售拿着「转客服」，客服拿着「转销售」——
# 谁也拿不到别人的业务工具，这就是交接模式的能力边界。
sales_agent = create_agent(
    model=llm,
    tools=[transfer_to_support],
    system_prompt="你是一名销售智能体。负责处理销售咨询。如果用户询问技术问题或售后支持，请转接给客服智能体。",
)

support_agent = create_agent(
    model=llm,
    tools=[transfer_to_sales],
    system_prompt="你是一名客服智能体。负责处理技术问题。如果用户询问价格或购买事宜，请转接给销售智能体。",
)

In [ ]:
# ---------- 4. 定义调用智能体的节点函数 ----------
def call_sales_agent(state: MultiAgentState) -> Command:
    """调用销售智能体的节点。"""
    # 子图返回的若是一个 Command（转接工具产生的），它会一路冒泡成节点的返回值，
    # 于是父图就按 Command 里的 goto 跳走 —— 交接就是这么发生的。
    response = sales_agent.invoke(state)
    return response


def call_support_agent(state: MultiAgentState) -> Command:
    """调用客服智能体的节点。"""
    # 与 call_sales_agent 完全对称：节点函数只做转发，
    # 「跳去哪」由子图里的 Command 决定 —— 所以这里没有 if/else 路由代码。
    response = support_agent.invoke(state)
    return response

In [ ]:
# ---------- 5. 定义路由函数（判断是结束还是继续） ----------
def route_after_agent(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent", "__end__"]:
    """根据活跃智能体进行路由，如果智能体没有调用工具则结束。"""
    messages = state.get("messages", [])

    # 检查最后一条消息：如果是不带工具调用的AI消息，说明对话结束
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
            return "__end__"

    # 否则路由到当前活跃的智能体
    active = state.get("active_agent", "sales_agent")
    return active if active else "sales_agent"


def route_initial(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent"]:
    """初始路由：根据状态中的活跃智能体决定，默认销售智能体。"""
    return state.get("active_agent") or "sales_agent"

In [ ]:
# ---------- 6. 构建状态图 ----------
builder = StateGraph(MultiAgentState)
builder.add_node("sales_agent", call_sales_agent)
builder.add_node("support_agent", call_support_agent)

# 起始边：根据初始状态进行条件路由
builder.add_conditional_edges(START, route_initial, ["sales_agent", "support_agent"])

# 智能体节点后的边：检查是否结束或跳转到另一个智能体
builder.add_conditional_edges("sales_agent", route_after_agent, ["sales_agent", "support_agent", END])
builder.add_conditional_edges("support_agent", route_after_agent, ["sales_agent", "support_agent", END])

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
# 配置线程 ID (用于区分不同的对话会话)
config = {"configurable": {"thread_id": "user_conversation_1"}}

In [ ]:
# ---------- 7. 便捷函数：抽出一轮的「谁值班 / 有没有发生转接」，便于观察 ----------
def describe_turn(result: dict) -> None:
    """打印本轮轨迹里的人机消息与转接痕迹。"""
    print("  本次对话内部通信：")
    for msg in result["messages"]:
        kind = type(msg).__name__
        if isinstance(msg, AIMessage) and msg.tool_calls:
            print(f"    {kind:<12} 要求调用 {[c['name'] for c in msg.tool_calls]} ← 交接动作")
        else:
            print(f"    {kind:<12} {str(msg.content)[:70]}")

    transfers = [
        msg for msg in result["messages"]
        if isinstance(msg, ToolMessage) and "转接" in str(msg.content)
    ]
    if transfers:
        print(f"  ✅ 发生了交接：{transfers[-1].content}")
    else:
        print("  本轮没有发生交接（当前值班的 Agent 自己处理完了）。")

In [ ]:
# 课案原文的第一问：技术问题 → 应当由客服接手
result = graph.invoke(
    {"messages": [{"role": "user", "content": "你好，我的账户登录有问题，能帮忙吗？"}]},
    config,
)
# 打印结果消息
for msg in result["messages"]:
    msg.pretty_print()
print("=== 多轮对话已启动 (输入 'exit' 或 'quit' 退出) ===")

# 无头执行时 stdin 不是终端 → isatty() 为 False → 自动跑两轮演示（等价于课案的交互循环）
interactive_ok = False
if sys.stdin.isatty():
    interactive_ok = True          # 先假定能交互，真读不到时由下面的 except 翻回来
    try:
        # ---------- 课案原文的交互循环 ----------
        while True:
            user_input = input("\nYou: ")
            if user_input.lower() in ["exit", "quit"]:
                print("=== 对话结束 ===")
                break
            result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config)
            print('本次对话内部通信')
            for msg in result["messages"]:
                print(msg)
            last_msg = result["messages"][-1]
            print(f"\nAI: {last_msg.content}")
    except EOFError:
        print("\n[提示] 读不到交互输入（无可用控制台），改用自动演示。\n")
        interactive_ok = False

if not interactive_ok:
    # ---------- 非交互环境的等价演示（避免 input() 卡死 / EOFError） ----------
    print("\n[非交互环境] 自动演示两轮，代替手工输入。\n")
    describe_turn(result)

    # 第二轮故意问销售问题：客服应当调用 transfer_to_sales 把对话交出去
    print("\n--- 第二轮：用户改问价格（预期触发 交接 → sales_agent） ---")
    before = len(result["messages"])
    result = graph.invoke(
        {"messages": [{"role": "user", "content": "你们这个套餐多少钱？我想买，能给我介绍下价格吗？"}]},
        config,
    )
    for msg in result["messages"][before:]:
        msg.pretty_print()
    describe_turn({"messages": result["messages"][before:]})
    print("\nAI:", result["messages"][-1].content)

### 预期输出

```text
================================ Human Message =================================
你好，我的账户登录有问题，能帮忙吗？
================================ Ai Message =================================
你好！...这属于技术支持范畴，我帮你转接给客服智能体...
Tool Calls:
  transfer_to_support (...)
================================ Tool Message =================================
已从销售智能体转接至客服智能体
=== 多轮对话已启动 (输入 'exit' 或 'quit' 退出) ===

[非交互环境] 自动演示两轮，代替手工输入。

  本次对话内部通信：
    HumanMessage 你好，我的账户登录有问题，能帮忙吗？
    AIMessage    要求调用 ['transfer_to_support'] ← 交接动作
    ToolMessage  已从销售智能体转接至客服智能体
    AIMessage    你好！我是客服智能体...
  ✅ 发生了交接：已从销售智能体转接至客服智能体

--- 第二轮：用户改问价格（预期触发 交接 → sales_agent） ---
  本次对话内部通信：
    AIMessage    要求调用 ['transfer_to_sales'] ← 交接动作
    ToolMessage  已从客服智能体转接至销售智能体
  ✅ 发生了交接：已从客服智能体转接至销售智能体

AI: ...（销售回复正文）
```

> ⚠️ 模型措辞、是否发动转接都**每次不同**。有个值得一看的细节：`route_initial` 默认路由到
> `sales_agent`，销售看到技术问题又转回客服——所以**第一轮就演示了一次「销售 → 客服」的交接**，
> 第二轮才是「客服 → 销售」。两轮的 `✅ 发生了交接` 都是确定性打印（只要模型真的调了转接工具）。
> 上面是实测值。

## 3. 路由与合并（Router + Fan-out/Fan-in）

交接是「任何时刻只有一个 Agent 在说话」；路由则是「**同时**让多个 Agent 分头干活，再把结果合并」。
它的图是一个三段式：

```mermaid
graph LR
    A["用户问题"] --> B["classify 分类"]
    B -->|"Send"| C["baidu"]
    B -->|"Send"| D["gitee"]
    C --> E["synthesize 合并"]
    D --> E
    E --> F["答案"]
```

三个概念是本节的骨架：
- **路由**：模型把「一个问题」拆成「N 个针对不同知识源的子问题」；
- **扇出（fan-out）**：`Send(节点名, 输入)` 列表 = 一次并行派发多个节点；
- **扇入（fan-in）**：`Annotated[list[...], operator.add]` 归约器把多路结果**相加合并**，而不是互相覆盖。

### 3.1 课案原版：三专家并行 + 归约器

原版最短：`router` 之后三条普通边同时指向 `tech / biz / risk` 三个专家（LangGraph 会自动并行），
三个专家各写一条 `opinions` 到同一个 `Annotated[list[str], operator.add]` 字段，
`merge` 节点等三者都完成后把意见拼起来，再让模型给一句总结。

In [ ]:
# ---------- 3.1 课案原版：路由与合并（三专家并行 + 归约器） ----------
import operator
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


class State(TypedDict):
    question: str
    # Annotated + add：多个专家的输出会「合并」到同一个列表里（fan-in 关键）
    opinions: Annotated[list[str], operator.add]
    summary: str


# ---------- 路由：拆任务 ----------
def router(state: State) -> dict:
    """调度节点：准备分发给专家的任务"""
    print(f"[路由] 分发任务：{state['question']}")
    return {}


# ---------- 三个专家（并行节点） ----------
def tech_expert(state: State) -> dict:
    r = llm.invoke(f"从技术角度一句话评价：{state['question']}")
    return {"opinions": [f"技术专家：{r.content}"]}


def biz_expert(state: State) -> dict:
    r = llm.invoke(f"从商业角度一句话评价：{state['question']}")
    return {"opinions": [f"商业专家：{r.content}"]}


def risk_expert(state: State) -> dict:
    r = llm.invoke(f"从风险角度一句话评价：{state['question']}")
    return {"opinions": [f"风险专家：{r.content}"]}


# ---------- 合并：汇总 ----------
def merge(state: State) -> dict:
    joined = "\n".join(state["opinions"])
    r = llm.invoke(f"综合以下多方意见，给出一句总结：\n{joined}")
    return {"summary": r.content}

In [ ]:
builder = StateGraph(State)
builder.add_node("router", router)
builder.add_node("tech", tech_expert)
builder.add_node("biz", biz_expert)
builder.add_node("risk", risk_expert)
builder.add_node("merge", merge)

builder.add_edge(START, "router")
# fan-out：router 之后三条边 = 三个专家并行执行
builder.add_edge("router", "tech")
builder.add_edge("router", "biz")
builder.add_edge("router", "risk")
# fan-in：三条汇入边，等全部专家完成后再进 merge
builder.add_edge("tech", "merge")
builder.add_edge("biz", "merge")
builder.add_edge("risk", "merge")
builder.add_edge("merge", END)

graph = builder.compile()

In [ ]:
result = graph.invoke({"question": "用 Rust 重写核心服务是否值得？", "opinions": []})
print("\n".join(result["opinions"]))
print("综合结论：", result["summary"])

### 预期输出

```text
[路由] 分发任务：用 Rust 重写核心服务是否值得？
商业专家：从商业角度，用 Rust 重写核心服务只有...
风险专家：从风险角度看，只有当团队具备成熟 Rust 能力...
技术专家：是否值得取决于核心服务的瓶颈是否真在内存安全...
综合结论： 综合来看，只有核心服务瓶颈确实在...
```

> ⚠️ 三个专家的评价与综合结论都是模型生成，措辞每次不同；而且**三个专家的打印顺序不固定**
> （并行执行，谁先跑完谁先进 `opinions`）。只有 `[路由] 分发任务：...` 是确定性打印。
> 注意 `opinions` 恰好三条、且都进了 `综合结论` 的输入——这正是 `operator.add` 归约器生效的证据。
> 上面是实测值。

### 3.2 完整版：结构化分类器 + `Send` 扇出

完整版把「三个硬编码专家」升级成「**分类器 + 动态扇出**」：分类器用结构化输出
`response_format=ClassificationResult` 决定「查哪几个知识源、各自用什么子问题」，
下游 `route_to_agents` 把分类结果变成 `list[Send]` 并行派发。

这里要特别说明一条**思考模型限制**（这是本节的降级点，原样保留）：

> ⚠️ 分类器用 `create_agent(response_format=ClassificationResult)` 做结构化路由决策，
> 思考模型端点**不支持强制 `tool_choice`**（会返回
> `400 Thinking mode does not support this tool_choice`）。源文件已内置降级：
> 分类失败时**两个知识源都查**（宁可多查不漏），`Send` 并行派发 + 归约器合并的演示不受影响。

另外两点课案特意标注：
- **`Send` 与 `Command.PARENT` 的区别**：`Send` 管「分头干活」（一批并行分支、各自带输入），
  `Command.PARENT` 管「换人接着干」（一跳一个目标、从子图冒泡回父图）。本节用 `Send`，
  因为两个知识源要**同时**查再汇总；交接（第 2 节）用 `Command.PARENT`，因为任何时刻只有一个 Agent 说话。
- **MCP 协议级错误（如 404 McpError）不是 `ToolException`**：`langchain-mcp-adapters`
  会故意让它传播并中断整个智能体，所以要用 `@wrap_tool_call` 兜成一条错误 `ToolMessage`，
  让模型看到失败原因后换招。

In [ ]:
# ---------- 3.2a 完整版：路由与合并（结构化分类器 + Send 扇出） ----------
import asyncio
import operator

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field
from typing_extensions import Annotated, Literal, TypedDict
from config import settings

try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    MCP_IMPORT_ERROR = None
except ImportError as exc:                      # pragma: no cover - 取决于本机环境
    MultiServerMCPClient = None
    MCP_IMPORT_ERROR = exc

In [ ]:
# ---------- 3.2b 状态与结构化输出模型（课案原文） ----------
class AgentInput(TypedDict):
    """每个子代理的简单输入状态。

    注意它**不是**父图的 RouterState：这是 Send 派发给子节点时单独构造的输入，
    只带一个 query（该节点专属的子问题），所以子节点看不到父图的其他键。
    """
    query: str


class AgentOutput(TypedDict):
    """每个子代理的输出。"""
    source: str
    result: str


class Classification(TypedDict):
    """单个路由决策：调用哪个代理以及使用什么查询。

    source 用 Literal 而不是 str 是有原因的：`Send(c["source"], ...)` 的节点名必须
    与图里注册的节点名**精确一致**，Literal 既让模型只能填这两个值，
    也让静态检查能发现拼写错误（写 "baidu_search" 就会跳到一个不存在的节点）。
    """
    source: Literal["baidu", "gitee"]
    query: str


class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    # Annotated + operator.add：多个并行节点写同一个键时用「列表相加」合并，而不是互相覆盖。
    # 这就是 fan-in（扇入）能拿到全部结果的原因。
    results: Annotated[list[AgentOutput], operator.add]  # 归约器收集并行结果
    final_answer: str


# 定义分类器的结构化输出模式
class ClassificationResult(BaseModel):
    """将用户查询分类为特定代理子问题的结果。"""
    classifications: list[Classification] = Field(
        description="要调用的代理列表及其针对性的子问题"
    )

In [ ]:
# ---------- 3.2c 把 MCP 工具异常降级为错误 ToolMessage（课案原文） ----------
@wrap_tool_call
async def mcp_error_handler(request, handler):
    """把 MCP 工具异常降级为错误 ToolMessage。

    MCP 协议级错误（如查询不存在的仓库/Issue 返回 404 McpError）不属于
    ToolException，langchain-mcp-adapters 会有意让它传播并中断整个智能体，
    这里统一捕获，让模型看到失败原因后改用其他工具继续。
    """
    try:
        # handler(request) 才是真正执行工具的那一句：
        # 正常返回 = 直接放行，异常 = 在这里「翻译」成一条 status="error" 的 ToolMessage。
        # 为什么用 ToolMessage 而不是重新 raise：模型看不到 Python 异常，
        # 但能看懂 ToolMessage 的内容 —— 它会据此换一个参数或换一个工具继续，而不是整个 Agent 崩掉。
        return await handler(request)
    except Exception as e:
        # content 里那句「不要重复相同的失败调用」是写给**模型**看的提示词，不是给人看的日志，
        # 删掉它模型很容易原地重试同一次失败调用。
        return ToolMessage(
            content=f"工具 {request.tool_call['name']} 调用失败: {e}。请检查参数（仓库名/Issue 编号必须来自搜索结果中真实存在的条目），修正后重试；不要重复相同的失败调用。",
            tool_call_id=request.tool_call["id"],
            name=request.tool_call["name"],
            status="error",
        )

In [ ]:
# ---------- 3.2d 工具来源：真 MCP（课案原文）/ 本地假工具（降级） ----------
# 百度千帆「联网搜索 MCP」与 Gitee MCP 的服务地址（课案原文）
BAIDU_MCP_URL = "https://qianfan.baidubce.com/v2/tools/web-search/mcp"
GITEE_MCP_URL = "https://api.gitee.com/mcp"


def preflight() -> tuple[bool, str]:
    """返回 (能否走真 MCP, 中文说明)。"""
    if MCP_IMPORT_ERROR is not None:
        return False, (
            "未安装 MCP 适配器：请先执行\n"
            '    uv add "mcp>=1.9,<2.0"\n'
            "    uv add langchain_mcp_adapters"
        )
    missing = []
    if not settings.baidu_qfan_api_key:
        missing.append("settings.baidu_qfan_api_key（百度千帆，用于联网搜索 MCP）")
    if not settings.gitee_api_key:
        missing.append("settings.gitee_api_key（Gitee，用于代码仓库 MCP）")
    if missing:
        return False, (
            "以下密钥为空，直连远程 MCP 会失败：\n    - " + "\n    - ".join(missing) +
            "\n    配置方式：在项目根目录 .env 里补上 BAIDU_QFAN_API_KEY=... / GITEE_API_KEY=...\n"
            "    本节将改用「本地假工具」替身，图结构一字不改地跑通演示。"
        )
    return True, "已检测到百度千帆 / Gitee 密钥，将直连两个远程 MCP 服务。"


async def build_real_mcp_tools():
    """课案原文：分别建两个 MCP 客户端，各自 get_tools()。"""
    # 创建 MCP 客户端，加载搜索工具
    # 注意两个客户端是**分别建**的：一个 MCP 服务端 = 一个 client 键，
    # 所以「两个知识源」在这里就是两个 client、两次 get_tools()。
    baidu_client = MultiServerMCPClient({
        "web-search-mcp-server": {
            "url": BAIDU_MCP_URL,
            "transport": "streamable-http",
            # 课案原文是 "Bearer xxx" 占位符；这里从配置读，绝不硬编码密钥
            "headers": {
                "Authorization": f"Bearer {settings.baidu_qfan_api_key}"
            },
        }
    })
    baidu_tools = await baidu_client.get_tools()

    # 创建 Gitee MCP 客户端
    gitee_client = MultiServerMCPClient({
        "gitee": {
            "url": GITEE_MCP_URL,
            "transport": "streamable-http",
            "headers": {
                "Authorization": f"Bearer {settings.gitee_api_key}"
            }
        }
    })
    gitee_tools = await gitee_client.get_tools()

    print(f"\n百度搜索工具: {len(baidu_tools)} 个")
    print(f"Gitee 工具: {len(gitee_tools)} 个")
    return baidu_tools, gitee_tools

In [ ]:
# ---------- 降级替身：名字和职责都对着课案里的 MCP 工具 ----------
@tool
async def baidu_search(query: str) -> str:
    """百度 AI 搜索：查找网络上的最新信息、技术文档、教程、新闻。

    query：搜索词或问题。
    说明：这是**降级演示用的本地假工具**（不联网），真实版本是百度千帆的联网搜索 MCP。
    """
    print(f"  [baidu_search] 收到查询：{query}")
    return (
        "【百度搜索结果】2025 年热度较高的数字人开源项目包括：\n"
        "1. HeyGem（硅基智能开源，唇形同步与数字人视频生成，GitHub 星标增长很快）；\n"
        "2. Duix.Heygem / duix.ai（实时交互数字人，支持本地部署）；\n"
        "3. Fay（数字人助理框架，侧重语音对话与知识库接入）；\n"
        "4. SadTalker / Wav2Lip（偏学术的说话人脸生成方案，常被二次开发）。"
    )


@tool
async def gitee_search_repositories(query: str) -> str:
    """搜索 Gitee 上的开源仓库，返回仓库名、语言与简介。

    query：搜索词。
    说明：这是**降级演示用的本地假工具**（不联网），真实版本是 Gitee 的代码仓库 MCP。
    """
    print(f"  [gitee_search_repositories] 收到查询：{query}")
    return (
        "【Gitee 搜索结果】\n"
        "1. guiji2025/heygem.ai —— Python，数字人视频合成，星标 1.2k；\n"
        "2. guiji2025/fay —— Python，数字人助理框架，星标 8k+；\n"
        "3. duixcom/Duix.Heygem —— C++/Python，实时数字人 SDK，星标 3k+。"
    )


def build_local_tools():
    """降级路径：一对本地假工具，接口签名与 MCP 工具一致（都是 async + 单字符串入参）。

    返回的是「两个工具列表」而不是一个：这样才能原样塞进下面 create_agent(tools=...) 的位置，
    让真 MCP 与降级路径走**同一行**组装代码 —— 图结构和路由逻辑一个字都不用改。
    """
    return [baidu_search], [gitee_search_repositories]

In [ ]:
# ---------- 3.2e 主流程 ----------
async def main() -> None:
    # ---------- 4.1 前置检查 ----------
    can_use_mcp, message = preflight()
    print("=" * 70)
    print("前置检查：", "通过" if can_use_mcp else "未通过")
    print(message)
    print("=" * 70)

    # ---------- 4.2 初始化模型 ----------
    # 课案原文：model = ChatOpenAI(model="deepseek-chat", ...)
    # 说明：课案这里写死 deepseek-chat，是因为「分类器必须用非思考模型」；
    #       本项目 settings.model_name 就是可用的对话模型，所以统一从 settings 取。
    model = init_chat_model(
        model_provider="openai",
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )

    # ---------- 4.3 取工具（真 MCP 或降级替身） ----------
    if can_use_mcp:
        try:
            baidu_tools, gitee_tools = await build_real_mcp_tools()
        except Exception as exc:
            print(f"[降级] 连接远程 MCP 失败：{type(exc).__name__}: {str(exc)[:160]}")
            print("       改用本地假工具，图结构与调用逻辑不变。\n")
            baidu_tools, gitee_tools = build_local_tools()
    else:
        baidu_tools, gitee_tools = build_local_tools()
        print(f"使用本地假工具：百度 {len(baidu_tools)} 个 / Gitee {len(gitee_tools)} 个\n")

    # ---------- 4.4 创建两个领域代理（课案原文，含 mcp_error_handler） ----------
    baidu_agent = create_agent(
        model,
        tools=baidu_tools,
        # 两个 Agent 挂的是**同一个** mcp_error_handler 实例 —— 无状态，可安全共享
        middleware=[mcp_error_handler],
        system_prompt=(
            "你是百度搜索专家。使用 AI 搜索功能查找网络上的最新信息，"
            "回答关于技术文档、新闻、教程等的一般性问题。"
        ),
    )

    gitee_agent = create_agent(
        model,
        tools=gitee_tools,
        middleware=[mcp_error_handler],
        system_prompt=(
            "你是 Gitee 专家。通过搜索 Gitee 上的代码仓库、Issue 和 Pull Requests，"
            "回答关于代码实现、API 参考和开发细节的问题。"
            "优先使用 search_open_source_repositories 搜索仓库；"
            "仅对搜索结果中确认存在的仓库查询其 Issue/Pull Requests，不要猜测仓库名或 Issue 编号。"
        ),
    )

    # ---------- 4.5 节点一：分类（用结构化输出做路由决策） ----------
    async def classify_query(state: RouterState) -> dict:
        """分类查询并确定要调用哪些代理。"""
        # 课案原文：classifier_model = ChatOpenAI(model="deepseek-chat", ...)
        classifier_model = init_chat_model(
            model_provider="openai",
            model=settings.model_name,
            api_key=settings.api_key,
            base_url=settings.base_url,
        )
        # 使用 response_format 参数创建代理
        classifier = create_agent(
            classifier_model,
            response_format=ClassificationResult,
            system_prompt="""分析此查询并确定要查询哪些知识源。
对于每个相关来源，生成针对该来源优化的子问题。

可用来源：
- baidu：百度搜索 - 网络搜索、最新资讯、技术文档、教程
- gitee：Gitee - 代码仓库、实现细节、API 参考、Issue、Pull Requests

仅返回与查询相关的来源。每个来源都应有针对该特定知识领域优化的子问题。

示例："我如何对 API 请求进行身份验证？"
- baidu："搜索 API 身份验证的最佳实践和教程"
- gitee："搜索身份验证相关的代码实现和示例" """
        )

        try:
            result = await classifier.ainvoke({
                "messages": [{"role": "user", "content": state["query"]}]
            })
        except Exception as exc:
            # 端点不支持结构化输出时的降级。分类器内部靠强制 tool_choice 拿结构化结果，
            # 思考模型端点会返回 `400 Thinking mode does not support this tool_choice`。
            # 降级策略保守：两个知识源都查（宁可多查，不要漏）。
            print(f"  [降级] 结构化分类不可用（{type(exc).__name__}）：{str(exc)[:110]}")
            print("         原因：分类器要强制 tool_choice，当前端点的模型是思考模式，不支持。")
            print("         兜底：本轮到两个知识源都查（并行派发与合并的演示不受影响）。")
            return {"classifications": [
                {"source": "baidu", "query": state["query"]},
                {"source": "gitee", "query": state["query"]},
            ]}

        # 直接取属性 `.classifications`，不用 json.loads 也不用正则 ——
        # 这就是 response_format 的价值：拿到的是 Pydantic 对象，字段名由类型系统保证。
        return {"classifications": result["structured_response"].classifications}

    # ---------- 4.6 条件边：把分类结果变成并行派发 ----------
    def route_to_agents(state: RouterState) -> list[Send]:
        """根据分类结果分发给代理。"""
        # Send(节点名, 传给该节点的输入)：一条 Send = 一个并行分支。
        # 返回空列表时图不会分发（下方 synthesize 会给出「未找到结果」的兜底答案）。
        return [
            Send(c["source"], {"query": c["query"]})
            for c in state["classifications"]
        ]

    # ---------- 4.7 节点二、三：两个领域代理（并行执行） ----------
    async def query_baidu(state: AgentInput) -> dict:
        """查询百度搜索代理。"""
        result = await baidu_agent.ainvoke({
            "messages": [{"role": "user", "content": state["query"]}]
        })
        # 只取最后一条消息的正文：子 Agent 内部调了多少次 MCP 工具、中间消息长什么样，
        # 对父图来说都不重要 —— 这就是「Agent 当工具」时天然的信息封装。
        return {"results": [{"source": "baidu", "result": result["messages"][-1].content}]}

    async def query_gitee(state: AgentInput) -> dict:
        """查询 Gitee 代理。"""
        try:
            result = await gitee_agent.ainvoke({"messages": [{"role": "user", "content": state["query"]}]})
            return {"results": [{"source": "gitee", "result": result["messages"][-1].content}]}
        except Exception as e:
            # 单个分支失败不影响另一条分支：把失败原因也当成一条结果交给汇总节点。
            print(f"Gitee 查询出错: {e}")
            return {"results": [{"source": "gitee", "result": f"Gitee 查询失败: {str(e)}"}]}

    # ---------- 4.8 节点四：合并（fan-in） ----------
    async def synthesize_results(state: RouterState) -> dict:
        """将所有代理的结果组合成一个连贯的答案。"""
        if not state["results"]:
            return {"final_answer": "未从任何知识来源找到结果。"}

        # 格式化结果以供综合
        formatted = [
            f"**来自 {r['source']}：**\n{r['result']}"
            for r in state["results"]
        ]

        synthesis_response = await model.ainvoke([
            {
                "role": "system",
                "content": f"""综合这些搜索结果以回答原始问题："：{state['query']}"

- 组合来自多个来源的信息，避免冗余
- 突出最相关和最可操作的信息
- 注明来源之间的任何差异
- 保持回复简洁且条理清晰"""
            },
            {"role": "user", "content": "\n\n".join(formatted)}
        ])

        return {"final_answer": synthesis_response.content}

    # ---------- 4.9 构建工作流（课案原文的链式写法） ----------
    # 注意 ["baidu", "gitee"] 那个列表：它是**声明式**的，只是告诉 LangGraph
    # 「这个条件边可能去这两个节点」，真正派发几个由 Send 列表长度决定。
    workflow = (
        StateGraph(RouterState)
        .add_node("classify", classify_query)
        .add_node("baidu", query_baidu)
        .add_node("gitee", query_gitee)
        .add_node("synthesize", synthesize_results)
        .add_edge(START, "classify")
        .add_conditional_edges("classify", route_to_agents, ["baidu", "gitee"])
        .add_edge("baidu", "synthesize")
        .add_edge("gitee", "synthesize")
        .add_edge("synthesize", END)
        .compile()
    )

    # ---------- 4.10 运行示例查询 ----------
    result = await workflow.ainvoke({"query": "最热的数字人项目"})

    print("\n" + "=" * 60)
    print("原始查询:", result["query"])
    print("\n分类结果:")
    for c in result["classifications"]:
        print(f"  {c['source']}: {c['query']}")
    print(f"\n并行返回的结果条数: {len(result['results'])}"
          f"（来源：{[r['source'] for r in result['results']]}）")

    print("\n" + "=" * 60 + "\n")
    print("最终答案:")
    print(result["final_answer"])

In [ ]:
# 同上：已 nest_asyncio.apply()，这里原样 asyncio.run(main())。
asyncio.run(main())

### 预期输出

```text
======================================================================
前置检查： 通过
已检测到百度千帆 / Gitee 密钥，将直连两个远程 MCP 服务。
======================================================================

百度搜索工具: 1 个
Gitee 工具: 25 个
  [降级] 结构化分类不可用（OpenAIInvalidRequestError）：...Thinking mode does not support this tool_choice...
         原因：分类器要强制 tool_choice，当前端点的模型是思考模式，不支持。
         兜底：本轮到两个知识源都查（并行派发与合并的演示不受影响）。

============================================================
原始查询: 最热的数字人项目

分类结果:
  baidu: 最热的数字人项目
  gitee: 最热的数字人项目

并行返回的结果条数: 2（来源：['baidu', 'gitee']）

============================================================

最终答案:
...（综合 Gitee/Baidu 真实搜索结果后的答案）
```

> ⚠️ 模型措辞每次不同；`[降级] 结构化分类不可用` 与「分类结果 / 结果条数」是**确定性打印**。
> 注意：本机密钥**有效**，所以两个真 MCP 都加载成功（百度 1 个、Gitee 25 个工具），
> 只有**分类器**那一步因思考模型不支持 `tool_choice` 走了降级；两个领域 Agent 用的是**真实搜索**。
> 密钥失效时才会打印 `[降级] 连接远程 MCP 失败` 并改用本地假工具。上面是实测值。

## 4. Skills 渐进披露（官方补充）

官方多 Agent 五模式的第 4 种，课案原本没覆盖。它的核心思想（官方 Key characteristics）：

- **提示词驱动的专业化**：技能本质是一段专业提示词 / 领域知识，不是代码；
- **渐进披露（progressive disclosure）**：技能**按需加载**，不加载就不占上下文。

注意和课案 `08_skills` 章的区别（容易混）：`08_skills` 讲的是 **DeepAgents 内置**的
skills（`SkillsMiddleware` + agentskills.io 规范，目录放好框架自动发现注入）；
**本文件讲的是 LangChain `create_agent` 侧的手工版**——官方原文明确说「内置 skill 支持见
Deep Agents」，LangChain 侧要自己搭一个 `load_skill` 工具。两条路线的关系是：
08 章是「用框架的能力」，本文件是「自己实现这套机制」。

In [ ]:
# ---------- 4a Skills 渐进披露（官方补充） ----------
import tempfile
from pathlib import Path

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from config import settings

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ================================================================
# 技能库：技能就是磁盘上的一个目录（SKILL.md + 可选附属资源）
# ================================================================
# 目录约定与课案 08_skills 章一致（agentskills.io 规范）：
#     skills/<技能名>/SKILL.md        ← 技能正文（YAML frontmatter + Markdown）
#     skills/<技能名>/assets/...      ← 附属资源，**用到时才读**
SKILLS: dict[str, str] = {
    "write_sql": """---
name: write_sql
description: SQL 查询编写专家：先给可执行 SQL，再解释思路
---

# SQL 编写技能

回答数据库问题时，严格按以下格式输出：

1. **SQL**：先给出一个可直接执行的 SQL 语句（用 ```sql 代码块）；
2. **解释**：再用不超过三句话说明思路（用到哪些表、为什么这样过滤）；
3. 不要臆造字段名；字段不确定时，先说明"需要看 schema"。

如需详细表结构，请读取附属文件：assets/schema.sql
""",
    "review_legal_doc": """---
name: review_legal_doc
description: 法务文档审阅：逐条列出风险点与修改建议
---

# 法务文档审阅技能

审阅合同时，输出一张风险清单，每条包含：

- **原文摘录**（不超过一行）
- **风险等级**（高 / 中 / 低）
- **修改建议**（一句话）

不要给笼统结论，必须逐条对应原文。
""",
}

SKILL_ASSETS: dict[tuple[str, str], str] = {
    ("write_sql", "assets/schema.sql"): """-- 简化的订单库表结构
CREATE TABLE users   (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
CREATE TABLE orders  (id INTEGER PRIMARY KEY, user_id INTEGER, amount REAL, created_at TEXT);
""",
}

In [ ]:
def build_skill_library(root: Path) -> None:
    """把技能写到磁盘（真实项目里这些文件本来就躺在仓库里）。"""
    for name, content in SKILLS.items():
        skill_dir = root / name
        skill_dir.mkdir(parents=True, exist_ok=True)
        (skill_dir / "SKILL.md").write_text(content, encoding="utf-8")
    for (name, rel), content in SKILL_ASSETS.items():
        asset = root / name / rel
        asset.parent.mkdir(parents=True, exist_ok=True)
        asset.write_text(content, encoding="utf-8")


def make_skill_tools(skills_root: Path) -> list:
    """按官方模式造两个工具：加载技能正文、读取技能的附属资源。"""

    @tool
    def load_skill(skill_name: str) -> str:
        """加载一个专业技能提示词。

        可用技能：
        - write_sql：SQL 查询编写专家
        - review_legal_doc：法务文档审阅

        返回该技能的完整提示词。
        """
        # 官方基础实现就是把技能正文读出来返回；技能名不合规时给出可用清单
        path = skills_root / skill_name / "SKILL.md"
        if not path.exists():
            return f"没有名为 {skill_name!r} 的技能。可用技能：{', '.join(SKILLS)}"
        return path.read_text(encoding="utf-8")

    @tool
    def read_skill_asset(skill_name: str, relative_path: str) -> str:
        """读取某个技能目录下的附属资源文件（如 assets/schema.sql）。"""
        path = (skills_root / skill_name / relative_path).resolve()
        # 安全边界：附属资源必须落在该技能目录内（防 ../ 越权读取）。
        if not path.is_relative_to((skills_root / skill_name).resolve()):
            return "路径越界，已拒绝"
        if not path.is_file():      # is_file 而不是 exists：目录也会让 exists 为真
            available = [rel for (n, rel) in SKILL_ASSETS if n == skill_name]
            return f"资源不存在。该技能可用的附属文件：{available or '（无）'}"
        return path.read_text(encoding="utf-8")

    return [load_skill, read_skill_asset]


SYSTEM_PROMPT = (
    "你是一个通用助手，可通过技能获得专业能力。\n"
    "可用技能：write_sql（SQL 编写）、review_legal_doc（法务审阅）。\n"
    "当用户的问题属于某个技能的领域时，先调用 load_skill 加载该技能的提示词，"
    "然后严格按技能里的要求作答；技能正文里提到的附属文件用 read_skill_asset 读取。"
)


def has_tool_call(result: dict, tool_name: str) -> bool:
    """检查这轮对话里模型是否真的调用了某个工具（用于兜底提示）。"""
    for message in result.get("messages", []):
        for call in getattr(message, "tool_calls", None) or []:
            if call["name"] == tool_name:
                return True
    return False


def final_text(result: dict) -> str:
    return str(result["messages"][-1].content)

In [ ]:
# ================================================================
# Demo 1：基础模式 —— 技能按需加载
# ================================================================
def demo_1_basic_skill_loading(skills_root: Path) -> None:
    print("=" * 70)
    print("Demo 1：基础模式 —— 模型自己决定加载哪个技能")
    print("=" * 70)

    agent = create_agent(
        model=model,
        tools=make_skill_tools(skills_root),
        system_prompt=SYSTEM_PROMPT,
    )
    result = agent.invoke({
        "messages": [{"role": "user", "content": "帮我查一下每个城市的订单总金额，写个 SQL"}],
    })

    if not has_tool_call(result, "load_skill"):
        print("  ⚠️ 本轮模型没有调用 load_skill（偶发行为），重跑一次通常就有；")
        print("     也可以把 system_prompt 里的要求写得更强制。")
        print("  模型直接回答：", final_text(result)[:120])
        return

    # 打印模型加载了哪个技能（工具消息里有技能正文）
    for message in result["messages"]:
        if message.type == "tool" and getattr(message, "name", "") == "load_skill":
            first_line = str(message.content).strip().splitlines()[0]
            print(f"  ✔ 模型调用了 load_skill，读到的技能开头：{first_line[:40]}")

    answer = final_text(result)
    print(f"  最终回答（前 200 字）：\n    {answer[:200].replace(chr(10), chr(10) + '    ')}")
    if "select" in answer.lower():
        print("  ✔ 回答里出现了 SQL（技能要求的格式生效了）")
    else:
        print("  （回答里没看到 SQL —— 模型可能没完全遵守技能格式，属模型行为）")
    print(
        "  ↑ 关键：技能正文是在**模型决定要用它之后**才进入上下文的。\n"
        "    用户问的是 SQL 问题时，法务技能一个字都没进上下文 —— 这就是渐进披露。"
    )

In [ ]:
# ================================================================
# Demo 2：渐进披露到底省了多少上下文（可量化）
# ================================================================
def demo_2_disclosure_savings() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：渐进披露省了多少上下文（量化对比）")
    print("=" * 70)

    # 方案 A：把所有技能全文塞进 system_prompt（"全量注入"）
    full_injection = SYSTEM_PROMPT + "\n\n" + "\n\n".join(SKILLS.values())
    # 方案 B：只列技能名与用途，正文按需加载（本文件的模式）
    def description_of(body: str) -> str:
        """从 SKILL.md 的 frontmatter 里取 description（按前缀找，不靠行号下标）。"""
        for raw_line in body.splitlines():
            if raw_line.strip().startswith("description:"):
                return raw_line.split("description:", 1)[1].strip()
        return "（无描述）"

    skill_index = "\n".join(f"- {name}：{description_of(content)}" for name, content in SKILLS.items())
    lazy = SYSTEM_PROMPT + "\n" + skill_index

    a, b = len(full_injection), len(lazy)
    print(f"  技能数量：{len(SKILLS)} 个")
    print(f"  方案 A 全量注入 system_prompt：{a} 字符")
    print(f"  方案 B 只列索引、正文按需加载：{b} 字符")
    print(f"  → 本轮省下 {a - b} 字符（约 {(a - b) / a:.0%}）")
    print(
        "  ↑ 技能越多差距越大（省的是**每轮请求**都要带的那份固定开销）；\n"
        "    渐进披露的代价是多一次工具调用（多一个来回）——\n"
        "    所以技能正文大的场景收益明显，技能很短时全量注入反而更省事。"
    )

In [ ]:
# ================================================================
# Demo 3：引用感知 —— 技能正文指向附属文件，用到时才读（第二级披露）
# ================================================================
def demo_3_reference_awareness(skills_root: Path) -> None:
    print("\n" + "=" * 70)
    print("Demo 3：引用感知 —— 技能指向附属文件，模型按需再读")
    print("=" * 70)

    agent = create_agent(
        model=model,
        tools=make_skill_tools(skills_root),
        system_prompt=SYSTEM_PROMPT,
    )
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "写个 SQL 统计每个城市的订单总金额。我不确定表结构，你先看 schema 再写。",
        }],
    })

    loaded = has_tool_call(result, "load_skill")
    read_asset = has_tool_call(result, "read_skill_asset")
    print(f"  加载技能 load_skill：{'✔' if loaded else '✗ 本轮没调'}")
    print(f"  读取附属文件 read_skill_asset：{'✔' if read_asset else '✗ 本轮没调'}")
    for message in result["messages"]:
        if message.type == "tool" and getattr(message, "name", "") == "read_skill_asset":
            print(f"    schema 内容（工具返回）：{str(message.content)[:60]}…")
    print(f"  最终回答（前 160 字）：{final_text(result)[:160]}")
    print(
        "  ↑ 两级披露：技能正文（第一级）→ 附属资源（第二级）。\n"
        "    如果模型这轮没读附属文件，是因为它觉得技能正文够了 —— 这是**模型判断**，\n"
        "    生产里可以在技能正文里把「必须先读 schema」写得更硬（或干脆用权限规则限制）。"
    )
    print(
        "\n  官方提到的另外两种扩展（本文件不展开，留作练习）：\n"
        "    · 动态工具注册：加载技能的同时注册新工具（技能正文 + 工具集一起变强）；\n"
        "    · 层级技能：技能里再定义子技能（data_science → pandas_expert / viz / stats）。"
    )

In [ ]:
# 技能写到临时目录：自包含、可重复运行，不往仓库里塞演示文件。
with tempfile.TemporaryDirectory(prefix="skills_demo_") as tmp:
    skills_root = Path(tmp) / "skills"
    build_skill_library(skills_root)
    print(f"技能库已生成到临时目录：{skills_root}")
    print(f"目录结构：{[str(p.relative_to(skills_root)) for p in sorted(skills_root.rglob('*')) if p.is_file()]}\n")

    demo_1_basic_skill_loading(skills_root)
    demo_2_disclosure_savings()
    demo_3_reference_awareness(skills_root)

print("\n全部 Demo 执行完毕。")

### 预期输出

```text
技能库已生成到临时目录：C:\Users\...\Temp\skills_demo_xxx\skills
目录结构：['review_legal_doc/SKILL.md', 'write_sql/SKILL.md', 'write_sql/assets/schema.sql']

Demo 1：基础模式 —— 模型自己决定加载哪个技能
  ✔ 模型调用了 load_skill，读到的技能开头：---
  最终回答（前 200 字）：
    ...（模型按 write_sql 技能给出的 SQL + 解释）
  ✔ 回答里出现了 SQL（技能要求的格式生效了）

Demo 2：渐进披露省了多少上下文（量化对比）
  技能数量：2 个
  方案 A 全量注入 system_prompt：589 字符
  方案 B 只列索引、正文按需加载：237 字符
  → 本轮省下 352 字符（约 60%）

Demo 3：引用感知 —— 技能指向附属文件，模型按需再读
  加载技能 load_skill：✔
  读取附属文件 read_skill_asset：✔
    schema 内容（工具返回）：-- 简化的订单库表结构…
  最终回答（前 160 字）：...

全部 Demo 执行完毕。
```

> ⚠️ 模型措辞、是否调用 `load_skill` / `read_skill_asset`、临时目录名（含随机后缀）都
> **每次不同**；`技能数量：2 个`、`方案 A/B` 的字符数、`✔/✗` 标记由代码确定但数值
> 随技能正文长度而定。上面是实测值。

## 5. 自定义工作流（官方补充）

官方五模式的最后一种，也是「LCEL 之死」的答案：官方新文档全站已不再以 LCEL 为主线，
**现在的编排手段就是 LangGraph 的 `StateGraph`**。官方原话（custom-workflow.mdx 核心洞察）：

> The core insight is that you can call a LangChain agent directly inside any LangGraph node.
> —— 任何 LangGraph 节点里都能直接调 `create_agent` 的产物。

官方还给了三种节点类型的框架（RAG pipeline 例子）：

| 节点类型 | 干什么 | 本 Demo 对应 |
|---|---|---|
| Model node（模型节点） | 用结构化输出改写查询 | `rewrite` |
| Deterministic node（确定性节点） | 检索，完全不经过 LLM | `retrieve`（本机无 embeddings，用关键词检索） |
| Agent node（智能体节点） | 带工具的 agent，负责推理 | `agent` |

以及「把一整个多 Agent 系统当作**一个节点**嵌进工作流」。

In [ ]:
# ---------- 5a 自定义工作流（官方补充） ----------
import re

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from config import settings

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def final_text(result: dict) -> str:
    """取结果里的最后一条消息文本。"""
    return str(result["messages"][-1].content)


def agent_answer(agent, prompt: str) -> str:
    """调用一个 agent 并返回它的最终回答（节点里复用的小工具函数）。"""
    result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return final_text(result)

In [ ]:
# ================================================================
# Demo 1：基础模式 —— agent 作为工作流节点
# ================================================================
class SimpleState(TypedDict):
    query: str
    answer: str


def build_simple_workflow():
    # 这个 agent 什么工具都不带：只是「模型 + 系统提示」的一层封装
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是简洁的助手，回答控制在两句话以内。",
    )

    def agent_node(state: SimpleState) -> dict:
        """LangGraph 节点：内部调用 LangChain agent。"""
        return {"answer": agent_answer(agent, state["query"])}

    return (
        StateGraph(SimpleState)
        .add_node("agent", agent_node)
        .add_edge(START, "agent")
        .add_edge("agent", END)
        .compile()
    )


def demo_1_agent_as_node() -> None:
    print("=" * 70)
    print("Demo 1：基础模式 —— agent 就是一个节点")
    print("=" * 70)

    workflow = build_simple_workflow()
    result = workflow.invoke({"query": "用一句话解释什么是 LangGraph"})
    print(f"  输入：用一句话解释什么是 LangGraph")
    print(f"  输出：{result['answer'][:150]}")
    print(
        "  ↑ 注意状态字段 answer 是**结构化**的：下游节点可以直接读它做判断，\n"
        "    而不是去解析模型的一整段自然语言 —— 这是把 agent 塞进流程的前提。"
    )

In [ ]:
# ================================================================
# Demo 2：官方的三节点 RAG 工作流（检索节点本地化）
# ================================================================
class RagState(TypedDict):
    question: str
    rewritten: str          # 改写后的查询
    keywords: list[str]     # 检索关键词
    documents: list[str]    # 命中的知识库文档
    answer: str


class RewrittenQuery(BaseModel):
    """改写节点的结构化输出（官方用 structured output 做这一步）。"""

    rewritten: str = Field(description="更适合检索的完整问句")
    keywords: list[str] = Field(description="3-5 个用于关键词匹配的词")


# 本地知识库：故意用中文短文档，方便人类核对检索结果对不对
KNOWLEDGE_BASE = [
    "LangGraph 的 StateGraph 用节点和边描述流程，节点是函数，边决定执行顺序。",
    "LangGraph 的 checkpointer 负责持久化，thread_id 用来区分不同会话，支持中断恢复。",
    "LangChain 的 create_agent 返回一个编译好的图，可以直接 invoke，也可以当作子图嵌入。",
    "DeepAgents 的 create_deep_agent 在 create_agent 基础上内置了文件系统、子代理与任务规划。",
    "LangSmith 提供链路追踪与评估，属于可观测性工具，与运行时框架解耦。",
]


def _bigrams(text: str) -> set[str]:
    """把文本切成字符二元组（中文无需分词器，且对措辞差异比整词匹配宽容）。"""
    cleaned = re.sub(r"[\s，。、？！：；（）【】“”‘’,.?!:;()\[\]]+", "", text.lower())
    return {cleaned[i:i + 2] for i in range(len(cleaned) - 1)}


def keyword_retrieve(query: str, keywords: list[str], top_k: int = 2) -> list[str]:
    """确定性检索：按字符 bigram 重叠数打分（不调用模型、不依赖 embeddings）。

    为什么不用整词匹配：中文里「状态存储」和文档里的「持久化」是同一件事但字面不同，
    整词匹配会全miss。bigram 重叠能抓住「会话」「存」这类共同片段 ——
    但它**治不了真正的语义鸿沟**（那正是向量检索存在的理由，而本机网关没有 embeddings）。
    """
    query_grams = _bigrams(query + "".join(keywords))
    scored: list[tuple[int, str]] = []
    for doc in KNOWLEDGE_BASE:
        overlap = len(query_grams & _bigrams(doc))
        if overlap:
            scored.append((overlap, doc))
    scored.sort(key=lambda item: -item[0])
    return [doc for _, doc in scored[:top_k]]


@tool
def get_current_date() -> str:
    """查询今天的日期（演示 agent 节点在检索之外补充实时信息）。"""
    from datetime import date

    return date.today().isoformat()

In [ ]:
def build_rag_workflow():
    structured_model = model.with_structured_output(RewrittenQuery)
    agent = create_agent(
        model=model,
        tools=[get_current_date],
        system_prompt=(
            "你是技术问答助手。只依据【参考资料】回答；"
            "资料里没有的内容就直说不知道，不要编造。回答控制在三句话内。"
        ),
    )

    def rewrite_node(state: RagState) -> dict:
        """① 模型节点：把口语化问题改写成检索友好的形式。"""
        try:
            out = structured_model.invoke([
                SystemMessage(
                    content="把用户问题改写成适合检索的形式，并提取关键词。"
                            "**必须全部用中文输出**（专有名词如 checkpointer 可保留英文）。"
                ),
                HumanMessage(content=state["question"]),
            ])
            print(f"    [rewrite] 改写为：{out.rewritten!r}，关键词：{out.keywords}")
            return {"rewritten": out.rewritten, "keywords": out.keywords}
        except Exception as exc:  # noqa: BLE001
            # 结构化输出偶发失败时退回原问题（工作流不该因为这一步挂掉）
            print(f"    [rewrite] 结构化输出失败（{type(exc).__name__}），退回原问题")
            return {"rewritten": state["question"], "keywords": []}

    def retrieve_node(state: RagState) -> dict:
        """② 确定性节点：关键词检索，完全不经过 LLM。"""
        docs = keyword_retrieve(state["rewritten"], state.get("keywords", []))
        print(f"    [retrieve] 命中 {len(docs)} 篇：{[d[:18] + '…' for d in docs]}")
        return {"documents": docs}

    def agent_node(state: RagState) -> dict:
        """③ 智能体节点：带着检索结果作答，必要时可调工具补充实时信息。"""
        context = "\n".join(f"- {doc}" for doc in state["documents"]) or "（没有命中任何资料）"
        prompt = f"【参考资料】\n{context}\n\n【问题】{state['question']}"
        return {"answer": agent_answer(agent, prompt)}

    return (
        StateGraph(RagState)
        .add_node("rewrite", rewrite_node)
        .add_node("retrieve", retrieve_node)
        .add_node("agent", agent_node)
        .add_edge(START, "rewrite")
        .add_edge("rewrite", "retrieve")
        .add_edge("retrieve", "agent")
        .add_edge("agent", END)
        .compile()
    )


def demo_2_rag_workflow() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：三节点 RAG 工作流（模型节点 → 确定性节点 → 智能体节点）")
    print("=" * 70)

    workflow = build_rag_workflow()

    # 同一套工作流跑两个问题，专门对比「确定性检索」的命中差异：
    #   问题 A 的用词和知识库接近（bigram 重叠多）→ 能命中；
    #   问题 B 是纯语义改写（口语问法）→ 可能命中不准，甚至答"不知道"。
    questions = [
        "checkpointer 和 thread_id 分别是干什么的？",
        "图里的状态是怎么存下来的？换会话会串吗？",
    ]
    for index, question in enumerate(questions, start=1):
        print(f"\n  --- 问题 {index}：{question} ---")
        result = workflow.invoke({"question": question})
        print(f"  最终回答：{result['answer'][:200]}")

    print(
        "\n  ↑ 两个问题问的其实是同一件事，但**确定性检索**只在用词接近时稳：\n"
        "    它可复现、免费、可单独测（给查询断言命中哪几篇），却治不了语义鸿沟；\n"
        "    向量检索正是为这个问题存在的 —— 而本机网关没有 embeddings（实测 503），\n"
        "    所以 RAG 类缺口在 官方文档缺口对照.md 里仍标着「待补（要 embeddings）」。\n"
        "    这也说明自定义工作流的一个实用价值：**检索质量可以脱离模型单独评估**。"
    )

In [ ]:
# ================================================================
# Demo 3：把一整个 Agent 当节点 + 条件分支与循环（evaluator-optimizer）
# ================================================================
class LoopState(TypedDict):
    topic: str
    draft: str
    score: int
    feedback: str
    attempts: int


REQUIRED_POINTS = ("文件系统", "子代理")   # 文案里必须覆盖的两个卖点
MAX_ATTEMPTS = 3


def build_loop_workflow():
    # 这个 agent 代表「被嵌入的多 Agent 系统」——换成课案 12/13 的多 Agent 产物同样成立
    writer = create_agent(
        model=model,
        tools=[],
        system_prompt=(
            "你是产品文案写手。用 2-3 句话写一段介绍，必须覆盖用户点名的所有卖点，"
            "并且直接给出文案本身，不要任何寒暄或解释。"
        ),
    )

    def generate_node(state: LoopState) -> dict:
        """生成节点：把（可选的）上一轮反馈拼进提示词，实现「带着意见重写」。"""
        attempts = state.get("attempts", 0) + 1
        prompt = f"请介绍 DeepAgents，卖点：{'、'.join(REQUIRED_POINTS)}。"
        if state.get("feedback"):
            prompt += f"\n上一版的问题是：{state['feedback']}，请修正。"
        draft = agent_answer(writer, prompt)
        # 为了在本 Demo 里**稳定看到回炉过程**，第一稿故意抹掉一个卖点。
        if attempts == 1:
            draft = draft.replace(REQUIRED_POINTS[-1], "相关能力")
            print(f"    [generate] （演示用：第一稿故意抹掉卖点「{REQUIRED_POINTS[-1]}」）")
        print(f"    [generate] 第 {attempts} 稿：{draft[:60]}…")
        return {"draft": draft, "attempts": attempts}

    def evaluate_node(state: LoopState) -> dict:
        """评分节点：确定性规则打分（0-100），并给出可读的改进意见。"""
        draft = state["draft"]
        hits = [point for point in REQUIRED_POINTS if point in draft]
        length_ok = 40 <= len(draft) <= 400
        score = int(100 * len(hits) / len(REQUIRED_POINTS)) - (0 if length_ok else 20)
        missing = [point for point in REQUIRED_POINTS if point not in draft]
        feedback = "" if not missing else f"缺少卖点：{'、'.join(missing)}"
        if not length_ok:
            feedback = (feedback + "；" if feedback else "") + f"篇幅不合适（当前 {len(draft)} 字）"
        print(f"    [evaluate] 得分 {score}，意见：{feedback or '无'}")
        return {"score": score, "feedback": feedback}

    def route_after_evaluate(state: LoopState) -> str:
        """条件边：达标或次数用尽就结束，否则回到生成节点重写。"""
        if state["score"] >= 100 or state["attempts"] >= MAX_ATTEMPTS:
            return END
        return "generate"

    return (
        StateGraph(LoopState)
        .add_node("generate", generate_node)
        .add_node("evaluate", evaluate_node)
        .add_edge(START, "generate")
        .add_edge("generate", "evaluate")
        # 条件边必须有通往 END 的出口，否则会无限循环（官方 Common Fixes 的第一条）
        .add_conditional_edges("evaluate", route_after_evaluate, ["generate", END])
        .compile()
    )


def demo_3_embedded_agent_and_loop() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：嵌入整个 Agent 当节点 + evaluator-optimizer 循环")
    print("=" * 70)

    workflow = build_loop_workflow()
    result = workflow.invoke({"topic": "DeepAgents", "draft": "", "score": 0, "feedback": "", "attempts": 0})
    print(f"  循环了 {result['attempts']} 轮，最终得分 {result['score']}")
    print(f"  最终文案：{result['draft'][:220]}")
    print(
        "  ↑ 循环的退出条件写在**条件边**里（达标 或 次数用尽 → END），\n"
        "    这是官方 Common Fixes 反复强调的一点：没有通往 END 的条件出口就会无限循环。\n"
        "    评分规则是确定性的，所以整个循环可复现；把 evaluate 换成模型（或 RubricMiddleware）\n"
        "    就得到官方 workflows-agents 里的 evaluator-optimizer 模式。"
    )

In [ ]:
demo_1_agent_as_node()
demo_2_rag_workflow()
demo_3_embedded_agent_and_loop()
print("\n全部 Demo 执行完毕。")

### 预期输出

```text
Demo 1：基础模式 —— agent 就是一个节点
  输入：用一句话解释什么是 LangGraph
  输出：...（两句话以内的解释）

Demo 2：三节点 RAG 工作流（模型节点 → 确定性节点 → 智能体节点）
  --- 问题 1：checkpointer 和 thread_id 分别是干什么的？ ---
    [rewrite] 结构化输出失败（OpenAIInvalidRequestError），退回原问题
    [retrieve] 命中 2 篇：['LangGraph 的 checkp…', 'LangChain 的 create…']
  最终回答：Checkpointer 负责持久化，thread_id 用来区分不同会话；配合起来支持中断恢复。
  --- 问题 2：图里的状态是怎么存下来的？换会话会串吗？ ---
    [rewrite] 结构化输出失败（OpenAIInvalidRequestError），退回原问题
    [retrieve] 命中 1 篇：['LangGraph 的 checkp…']
  最终回答：状态由 checkpointer 负责持久化保存，用 thread_id 区分不同会话...

Demo 3：嵌入整个 Agent 当节点 + evaluator-optimizer 循环
    [generate] （演示用：第一稿故意抹掉卖点「子代理」）
    [generate] 第 1 稿：...…
    [evaluate] 得分 50，意见：缺少卖点：子代理
    [generate] 第 2 稿：...…
    [evaluate] 得分 100，意见：无
  循环了 2 轮，最终得分 100
  最终文案：...

全部 Demo 执行完毕。
```

> ⚠️ 模型措辞、检索命中篇数、循环轮数都**每次可能不同**（Demo 3 的得分由确定性规则算出，
> 但轮数取决于模型第一稿是否被修掉卖点）。有个本机实测结论值得记：`with_structured_output`
> 在思考模型（deepseek-flash）上**同样失败**，所以 `[rewrite]` 每次都走「退回原问题」的兜底，
> 检索用的是原始问句。上面是实测值。

## 小结

官方把多 Agent 归纳为五种模式，按「控制权怎么流转」串起来：

| 模式 | 控制权 | 一句话记忆点 |
|---|---|---|
| ① 子代理 | 主 Agent 始终在场 | 子 Agent 的答复就是一坨文本 → 当工具 |
| ② 交接 | 换人接着干 | `Command(goto=..., graph=Command.PARENT)` |
| ③ 路由 | 分头干活再汇总 | `Send` 并行 + `operator.add` 归约 |
| ④ 技能 | 提示词按需加载 | 手写 `load_skill`，正文用到才进上下文 |
| ⑤ 自定义工作流 | Agent 当节点混编 | 任何 LangGraph 节点里都能调 `create_agent` |

一条主线：**「一个 Agent 当另一个 Agent 的工具」是原点**（①），往上加「控制权转移」
得到交接（②），加「并行」得到路由（③），加「提示词按需加载」得到技能（④），
最后把 Agent 彻底降格成「流程里的普通节点」得到自定义工作流（⑤）。

## 常见坑

1. **思考模型回放 AIMessage 会 400**（第 2.1 节）：DeepSeek 思考模式要求把
   `reasoning_content` 一起回传，而 AIMessage 经 LangGraph 状态往返后该字段丢失。
   降级：摘掉分诊台那条 AIMessage，只把用户消息交给专家。
2. **思考模型不支持强制 `tool_choice`**（第 3.2 节）：分类器用
   `create_agent(response_format=...)` 做结构化路由会 400。降级：两个知识源都查。
3. **子 Agent 的 tools 必须来自 `await client.get_tools()`**：忘了 await 会拿到
   coroutine 对象，`create_agent` 直接报类型错误。
4. **转接工具要成对回填消息**：`update={"messages": [last_ai_message, transfer_message]}`
   两条必须都在，只写 ToolMessage 会出现孤儿 ToolMessage，下次请求直接 400。
5. **`Send` 的目标名必须与节点名精确一致**：`Send("baidu_search", ...)` 不报错而是
   静默丢失；所以 `Classification.source` 用 `Literal` 限定取值。
6. **没有 `operator.add` 会丢结果**：把 `results: Annotated[list, operator.add]`
   写成普通 `list`，并行分支会互相覆盖，只剩一条（且不报错，最难查）。
7. **条件边一定要有通往 END 的出口**（第 5 节 Demo 3）：否则循环不收敛、无限循环。
8. **MCP 协议级错误不是 `ToolException`**：`langchain-mcp-adapters` 会故意让它传播
   并中断整个 Agent，要用 `@wrap_tool_call` 兜成错误 ToolMessage。

## 官方链接

- 多 Agent 总览：<https://docs.langchain.com/oss/python/langchain/multi-agent>
- 技能渐进披露：<https://docs.langchain.com/oss/python/langchain/multi-agent/skills>